# Classic RAG vs Graph RAG — Practical Comparison

Run **both** Classic RAG and Graph RAG on the **same complex document**, ask the **same questions**, and compare answers side by side. Graph RAG answers include the **subgraph** used to generate them.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Load API key from any .env location
for p in [Path("."), Path(".."), Path("../.."), Path("../../..")]:
    load_dotenv(p.resolve() / ".env")

# Neo4j connection
NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "workshop2024")

# Connect and clear all existing data
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    count = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
driver.close()

print(f"Neo4j connected. All data cleared. Nodes: {count}")
print(f"OpenAI key: {os.environ.get('OPENAI_API_KEY', 'NOT SET')[:15]}...")

In [ ]:
from pathlib import Path
import os, json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
import chromadb

load_dotenv(Path(".").resolve().parent / ".env")
load_dotenv(Path(".").resolve() / ".env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "workshop2024")

def ask_llm(prompt, system="You are a helpful assistant."):
    return llm.invoke([SystemMessage(content=system), HumanMessage(content=prompt)]).content

print("Setup complete!")

---
## The Document

A complex corporate dossier with multiple people, companies, timelines, acquisitions, lawsuits, and cross-references -- designed to stress-test RAG.

In [ ]:
document = """
MERIDIAN HEALTH TECHNOLOGIES — FULL CORPORATE DOSSIER

CHAPTER 1: ORIGINS (2016-2017)

In late 2016, Dr. Amara Osei was wrapping up her 12th year at Johnson & Johnson in New Brunswick, New Jersey, where she had risen to Senior Director of Computational Drug Discovery. Her team of 45 researchers had developed J&J's first AI-powered molecule screening platform, called MoleculeAI, which reduced early-stage drug candidate identification from 18 months to 3 months. During her time at J&J, Amara worked closely with Dr. Robert Zhang, a computational biologist who reported to her and co-invented three patents related to patient cohort matching algorithms.

Meanwhile, Dr. James Chen was a tenured professor of Biomedical Informatics at Harvard Medical School in Boston. James had published 89 papers on clinical data interoperability and had received a $2.1M NIH grant to study how electronic health records could be used to predict clinical trial eligibility. His graduate student, Dr. Sarah Kim, was completing her PhD dissertation on BERT-based medical text understanding.

Amara and James met at the MIT Healthcare AI Summit in October 2016, organized by Professor Regina Barzilay. During a panel discussion on "AI for Clinical Trials," they realized their work was complementary — Amara had the drug discovery expertise and industry connections, James had the clinical informatics knowledge and academic network. Over dinner at Legal Sea Foods in Cambridge, they sketched the idea for what would become Meridian Health Technologies on a napkin.

They incorporated Meridian Health Technologies in March 2017 in Delaware, with headquarters in Cambridge, Massachusetts, in a small office on Main Street near Kendall Square. Amara left J&J (forfeiting approximately $800K in unvested stock options) and James took a leave of absence from Harvard.

Their first hire was Dr. Lars Eriksson as Chief Technology Officer. Lars was a Swedish-American engineer who had spent 8 years at IBM Watson Health in Yorktown Heights, New York, where he had built the oncology treatment recommendation engine that was deployed in 14 hospitals across Asia. After IBM scaled back Watson Health in 2017, Lars spent 6 months at Flatiron Health in New York City (a cancer data analytics company later acquired by Roche for $1.9 billion in April 2018). Lars had grown frustrated with Flatiron's pivot away from AI toward data services, which is why he was receptive when Amara called.

CHAPTER 2: SEED FUNDING AND EARLY TEAM (2017-2018)

Meridian's first investor was Atlas Venture, a Boston-based life sciences venture fund. Dr. Priya Mehta, a partner at Atlas Venture and former Chief Medical Officer at Pfizer, personally championed the investment after seeing Amara present at the JP Morgan Healthcare Conference in January 2017. Atlas led the $8M seed round, with additional participation from Polaris Partners and angel investor Dr. Thomas Lee (co-founder of One Medical, which was acquired by Amazon for $3.9B in 2023).

Priya Mehta joined Meridian's board of directors. She brought invaluable pharmaceutical industry connections — during her 9 years at Pfizer, she had overseen 23 drug approvals and knew the heads of R&D at every major pharma company. She introduced Meridian to executives at Novartis, Roche, and AstraZeneca within the first year.

James Chen recruited his star PhD student, Dr. Sarah Kim, as the company's first research scientist. Sarah had just defended her dissertation on "Transformer-Based Clinical Text Understanding" and had received offers from Google Health, Amazon Lab126, and Apple Health. She chose Meridian because James promised her she could lead her own research team within a year.

The early team also included Dr. Kenji Tanaka, a data engineer hired from Palantir Technologies, where he had built healthcare data pipelines for the CDC during the 2017 flu season. Kenji designed Meridian's initial data infrastructure on AWS, using a combination of S3, Redshift, and custom ETL pipelines.

CHAPTER 3: FIRST PRODUCTS (2019-2020)

Meridian launched its first product, TrialMatch, in September 2019 after 18 months of development. TrialMatch used NLP to parse clinical trial protocols from ClinicalTrials.gov and match patients to eligible trials based on their electronic health records. The technology was built on BERT-based models fine-tuned on 2.3 million medical abstracts from PubMed.

Dr. Sarah Kim led the NLP research team of 8 people that developed TrialMatch. Her team's key innovation was a "medical entity linking" module that could map informal clinical notes ("the patient has sugar problems") to formal medical ontologies (ICD-10 code E11 for Type 2 Diabetes). This module achieved 91% accuracy on the n2c2 benchmark dataset, beating the previous state-of-the-art by 7 percentage points.

TrialMatch's first three customers were Massachusetts General Hospital (signed by Dr. David Reynolds, their Chief of Clinical Research), Cleveland Clinic (through an introduction by Priya Mehta's former Pfizer colleague Dr. Anna Brooks), and Mount Sinai Health System in New York (where James Chen had a long-standing academic collaboration with Dr. Eric Schadt, the Dean of the Icahn School of Medicine).

Meanwhile, Lars Eriksson's engineering team built MediGraph, an internal knowledge graph platform that connected drugs, diseases, genes, clinical trials, patient cohorts, and adverse events. MediGraph used Neo4j as its graph database backend and contained over 2.4 million nodes and 18 million relationships at launch. The knowledge graph could model complex biomedical relationships like "Drug A inhibits Protein B, which is encoded by Gene C, which is overexpressed in Disease D, which has a 23% co-occurrence with Disease E." This knowledge graph became the foundational data layer powering all Meridian products.

Kenji Tanaka built the data ingestion pipelines that fed MediGraph, pulling data from DrugBank, ChEMBL, the Human Gene Ontology, and the FDA Adverse Event Reporting System (FAERS). The pipeline processed approximately 50GB of new data weekly.

CHAPTER 4: SERIES A AND COMPETITIVE LANDSCAPE (2020-2021)

In January 2020, Meridian raised a $35M Series A led by Andreessen Horowitz. Ben Horowitz personally attended the board meeting and joined as a board observer. General partner Vijay Pande (who had previously founded Globavir Biosciences and ran Stanford's Folding@home project) took the board seat. Existing investor Atlas Venture participated in the round, along with new investors GV (Google Ventures), Khosla Ventures, and F-Prime Capital (the venture arm of Fidelity Investments).

This round was complicated by a conflict of interest: Atlas Venture was simultaneously evaluating an investment in BioNexus AI, a competing clinical trial matching platform. Dr. Priya Mehta, despite her personal loyalty to Meridian, felt she could not objectively serve on Meridian's board while her firm was considering a competing investment. She stepped down from the board in February 2020, though she retained her personal angel investment in Meridian.

BioNexus AI had been founded in 2019 by Dr. Robert Zhang, who had left Johnson & Johnson six months after Amara Osei's departure. Robert had been offered a co-founder role at Meridian by Amara, but he declined due to disagreements about equity split — Amara offered him 8% while Robert wanted 15%, arguing that the patient cohort matching patents he co-invented at J&J were foundational to Meridian's technology. Robert instead raised $12M from Atlas Venture and New Enterprise Associates (NEA) to build BioNexus AI in San Diego.

The competitive tension between Meridian and BioNexus became personal. Both companies were pitching to the same hospital systems, and Robert Zhang publicly claimed at the HIMSS conference in March 2020 that Meridian's TrialMatch was built on algorithms he had co-developed at J&J. Amara responded with a blog post stating that Meridian's technology was entirely original and that Robert's claims were "baseless and defamatory."

With the Series A funding, Meridian made several key hires:
- Dr. Maya Patel as VP of Clinical Affairs. Maya had spent 15 years at the FDA, most recently as Deputy Director of the Office of Biostatistics in the Center for Drug Evaluation and Research (CDER), where she personally oversaw the statistical review of 200+ drug applications. Her understanding of FDA submission requirements and her personal relationships with current FDA reviewers proved invaluable.
- Dr. Henrik Johansson as VP of Engineering, recruited from Spotify's Stockholm office where he had managed a 120-person engineering team. Henrik brought expertise in scaling distributed systems.
- Rachel Torres as VP of Marketing, previously the head of healthcare marketing at Salesforce Health Cloud.

CHAPTER 5: PRODUCT EXPANSION AND PARTNERSHIPS (2021-2022)

Meridian launched two new products in 2021:

DrugSight was a drug interaction prediction tool built on top of MediGraph. It could predict adverse drug interactions by traversing the knowledge graph to find indirect pharmacological pathways between medications that had never been studied together. For example, DrugSight could determine that Drug X (a blood thinner) and Drug Y (an antifungal) might interact dangerously because both affect the CYP3A4 liver enzyme, even though no clinical study had ever tested their combination. The product was co-developed with Novartis under a $4.5M joint development agreement signed by Novartis's Head of Digital Health, Dr. Simone Koehler. Novartis became both a development partner and the first enterprise customer. Pfizer and Roche signed on as customers by Q3 2021.

DrugSight's data backbone was enriched by Sarah Kim's NLP team, which extracted drug-gene-disease relationships from over 500,000 PubMed abstracts using a fine-tuned BioBERT model. This automated extraction process added approximately 340,000 new relationships to MediGraph.

ClinicalOS was an operating system for clinical trial management, developed in partnership with Veeva Systems (a cloud software company for the life sciences industry valued at $35B). The partnership was negotiated by James Chen, who had met Veeva's CEO Peter Gassner at a Stanford Healthcare Conference. ClinicalOS integrated TrialMatch's patient matching capabilities with real-time trial monitoring, site selection, and regulatory document management. Contract Research Organizations (CROs) including Parexel, IQVIA, and PPD signed as early adopters.

During this period, Dr. Kenji Tanaka was promoted to VP of Data Engineering and began building a real-time data streaming platform using Apache Kafka and Flink, replacing the batch-based ETL pipelines that were struggling to keep pace with MediGraph's growth (now 5.8 million nodes and 47 million relationships).

CHAPTER 6: THE BIONEXUS LAWSUIT AND LEADERSHIP CHANGES (2022-2023)

In June 2022, BioNexus AI filed a patent infringement lawsuit against Meridian Health Technologies in the United States District Court for the District of Delaware. The lawsuit (Case No. 22-cv-0847) alleged that Meridian's patient matching algorithm in TrialMatch was derived from U.S. Patent No. 10,892,417, titled "Computational Methods for Patient Cohort Identification," which Robert Zhang had co-invented with Amara Osei while both were employed at Johnson & Johnson. The patent had been assigned to J&J, which then licensed it exclusively to BioNexus AI as part of Robert's departure agreement.

The lawsuit created significant distraction and legal costs. Meridian's outside counsel, WilmerHale (a prominent Boston law firm), estimated legal fees of $5-8M through trial. The board debated whether to fight or settle. Ben Horowitz advocated for aggressive defense, while Vijay Pande favored settlement to preserve the company's focus on product development.

The case was eventually settled in March 2023, with Meridian paying $12M in damages and both companies cross-licensing certain patient matching patents. As part of the settlement, both companies agreed to a non-disparagement clause.

During this turbulent period, several leadership changes occurred:

Lars Eriksson departed as CTO in August 2022 to become CTO of Recursion Pharmaceuticals (a clinical-stage biotech company in Salt Lake City using AI for drug discovery). Lars had grown increasingly frustrated with what he perceived as the board's underinvestment in foundational technology. He also disagreed with James Chen's preference for partnering with established companies (like Veeva) rather than building proprietary technology. Lars's departure was a significant blow — he had architected MediGraph from scratch.

Dr. Henrik Johansson was considered for the CTO role but withdrew his candidacy, citing concerns about the company's technical debt. He left Meridian in October 2022 to return to Spotify as their VP of ML Engineering.

The board ultimately hired Dr. Wei Zhang as CTO in November 2022. Wei had spent 3 years at Anthropic in San Francisco, where he had led the team responsible for fine-tuning Claude for healthcare applications. Before Anthropic, Wei had been a research scientist at Google DeepMind in London, working on AlphaFold. Wei's hiring signaled Meridian's pivot toward large language models and away from the BERT-based approaches that Lars had championed.

Wei immediately began modernizing Meridian's technology stack. He replaced the BERT-based models in TrialMatch with a fine-tuned version of Claude (Anthropic's LLM), which improved patient matching accuracy from 78% to 94% on their internal benchmark of 10,000 patient-trial pairs. He also added RAG (Retrieval-Augmented Generation) capabilities to DrugSight, allowing clinicians to ask natural language questions about drug interactions and receive answers grounded in MediGraph's knowledge graph data. This combination of LLM + knowledge graph proved far more powerful than either approach alone.

CHAPTER 7: SERIES B AND GLOBAL EXPANSION (2023-2024)

Meridian raised a $120M Series B in June 2023, led by SoftBank Vision Fund 2, with participation from all existing institutional investors (Andreessen Horowitz, GV, Khosla Ventures, F-Prime Capital, and Atlas Venture). The round valued Meridian at $800M pre-money. SoftBank's Deep Nishar joined the board.

With the Series B capital, Meridian expanded internationally:
- London office: Hired Dr. Ingrid Weber from Bayer's Berlin office as Head of European Operations. Ingrid had 20 years of pharmaceutical industry experience and had previously served as Bayer's VP of Digital Health for EMEA.
- Munich office: Established to serve the DACH region (Germany, Austria, Switzerland), staffed with 15 employees.

Key European customers signed in 2023-2024:
- NHS England (contract worth £8.5M over 3 years for deploying TrialMatch across 12 NHS trusts)
- Charité University Hospital in Berlin (Germany's largest university hospital)
- AstraZeneca (global deployment of DrugSight across their R&D division)
- Roche Diagnostics (expanding beyond their existing DrugSight contract)

Amara Osei and James Chen co-authored a landmark paper in Nature Medicine (Vol. 29, pp. 2847-2859, 2023) titled "Knowledge Graph-Augmented Clinical Trial Matching: A Randomized Controlled Study Across 47 Clinical Sites." The study, conducted in collaboration with researchers from Harvard Medical School (Dr. Isaac Kohane), Stanford School of Medicine (Dr. Nigam Shah), and the Mayo Clinic (Dr. Hongfang Liu), demonstrated that Meridian's approach reduced patient screening time by 60% and increased trial enrollment rates by 40% compared to manual matching. The paper was cited 340 times within its first year.

CHAPTER 8: ACQUISITIONS AND RESTRUCTURING (2024)

In February 2024, Meridian acquired GenomicInsight, a 40-person genomics startup based in San Diego, for $45M in cash and stock. GenomicInsight had been founded in 2021 by Dr. Lisa Park (a computational genomics researcher from UC San Diego, no relation to any other individual mentioned in this document). GenomicInsight specialized in using graph neural networks to predict individual patient drug responses based on their genomic profile. Their technology complemented MediGraph by adding a genomics layer to Meridian's knowledge graph — connecting specific genetic variants to drug efficacy predictions. Dr. Lisa Park became Meridian's VP of Genomics and brought her entire team, including Dr. Yuki Sato, a leading expert in pharmacogenomics who had previously worked at 23andMe.

In September 2024, Meridian acquired PatientLink, a 25-person patient engagement platform based in Chicago, for $30M. PatientLink had developed a mobile app used by 50,000 clinical trial participants for appointment scheduling, symptom reporting, and medication adherence tracking. PatientLink's co-founder and CEO, Michael Torres, joined Meridian as VP of Patient Experience. The other co-founder, Dr. Angela Martinez, left after the acquisition to start a new company called HealthBridge in Austin, Texas.

CHAPTER 9: CURRENT LEADERSHIP AND STATE (2025)

As of early 2025, Meridian Health Technologies has undergone significant leadership evolution:

James Chen transitioned from CEO to Executive Chairman in January 2025, citing a desire to focus on the company's long-term scientific vision. The board hired Dr. Fatima Al-Hassan as the new CEO. Fatima had been Senior Vice President at Roche Diagnostics in Basel, Switzerland, for 11 years, most recently overseeing their $4.2B diagnostics division. She was recruited by executive search firm Spencer Stuart.

Dr. Maya Patel left Meridian in March 2024 to become Commissioner of the FDA — a historic appointment as the first person of Indian descent to hold the position. Her deep knowledge of both the regulatory and technology sides of healthcare made her a consensus nominee, confirmed by the Senate with an 89-11 vote.

Dr. Sarah Kim was promoted from Head of NLP Research to Chief Science Officer, taking over responsibilities previously shared between her and the departed Lars Eriksson. She now oversees a research team of 65 people.

Dr. Wei Zhang remains CTO and is leading the development of MediGraph 2.0, which will incorporate temporal knowledge graphs (tracking how medical knowledge evolves over time), multi-modal data integration (combining clinical notes, medical imaging, genomic data, and wearable device data), and federated learning capabilities (allowing hospitals to contribute data without sharing patient information).

Dr. Kenji Tanaka was promoted to VP of Platform Engineering and now manages a team of 40 engineers.

Rachel Torres (VP of Marketing) left in June 2024 to become CMO of Tempus AI (a precision medicine company founded by Eric Lefkofsky, valued at $6.1B).

The company now has 450 employees across Cambridge (HQ, 200 employees), San Francisco (100), London (80), Munich (40), and San Diego (30). Meridian serves 200+ hospitals and 15 pharmaceutical companies globally, with annual recurring revenue of $85M.

EPILOGUE: IRONIC CONNECTIONS

Priya Mehta, who had championed Meridian's seed investment at Atlas Venture and then stepped down from the board over the BioNexus conflict, ironically ended up joining BioNexus AI's board of directors in September 2024 — after BioNexus was acquired by Illumina (the genomics giant) for $500M. Priya now sits on the boards of both Illumina and two other biotech companies.

Robert Zhang, Meridian's erstwhile rival, was named CEO of the Year by BioPharma Dive in 2024, largely due to the successful Illumina acquisition. He and Amara Osei reportedly reconciled at the 2024 JP Morgan Healthcare Conference — the same event where Priya Mehta had first heard Amara's pitch seven years earlier.

Lars Eriksson's Recursion Pharmaceuticals went public in 2024 (NYSE: RXRX) at a $4.2B valuation. Lars was featured in Wired magazine's "25 People Shaping the Future of Medicine."

Dr. Thomas Lee, the angel investor from the seed round, saw his investment in Meridian grow from $250K to an estimated $12M at the Series B valuation — a 48x return.
"""

print(f"Document: {len(document)} characters, {len(document.split())} words")
print(f"That's about {len(document.split()) // 250} pages")
print(f"\nThis document has:")
print(f"  - 25+ people with complex interconnected career paths")
print(f"  - 15+ companies (employers, investors, acquirees, competitors)")
print(f"  - 3 funding rounds across different sections")
print(f"  - Lawsuits, board changes, departures, promotions")
print(f"  - People appearing in 3-4 different contexts each")
print(f"  - An epilogue that references events from the beginning")
print(f"  - Timeline spanning 2016-2025 across 9 chapters")

---
## Build BOTH systems on the same document

### System 1: Classic RAG (ChromaDB)

In [ ]:
# Realistic RAG settings — small chunks, low top_k
# This is how RAG works in production: you can't retrieve the whole document
CHUNK_SIZE = 300  # ~60 words per chunk
CHUNK_OVERLAP = 30
TOP_K = 2  # only 2 chunks retrieved per question

chunks = [document[i:i+CHUNK_SIZE] for i in range(0, len(document), CHUNK_SIZE - CHUNK_OVERLAP)]
print(f"Created {len(chunks)} chunks (size={CHUNK_SIZE}, top_k={TOP_K})")
print(f"Each question only sees {TOP_K * CHUNK_SIZE} chars out of {len(document)} total ({TOP_K * CHUNK_SIZE * 100 // len(document)}% of document)\n")

# Embed and store
chunk_embeddings = embeddings.embed_documents(chunks)

chroma = chromadb.Client()
try: chroma.delete_collection("rag_showdown")
except: pass
collection = chroma.create_collection("rag_showdown")
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=chunk_embeddings,
    documents=chunks,
)
print(f"Stored {collection.count()} chunks in ChromaDB")
print(f"\nKey insight: RAG only retrieves {TOP_K}/{collection.count()} chunks per question.")
print(f"Information spread across 9 chapters WILL be missed.")

### System 2: Graph RAG (Neo4j)

In [ ]:
# ── Schema ────────────────────────────────────────────────────────
class Entity(BaseModel):
    name: str
    type: str = Field(description="PERSON, COMPANY, PRODUCT, HOSPITAL, INVESTOR, CITY, UNIVERSITY, TECHNOLOGY")
    description: str

class Relationship(BaseModel):
    source: str
    target: str
    type: str
    description: str

class ChunkKG(BaseModel):
    entities: list[Entity] = Field(default_factory=list)
    relationships: list[Relationship] = Field(default_factory=list)


# ── Chunk the document for extraction (larger chunks than RAG) ────
# Real systems extract from overlapping chunks, then MERGE entities
extract_chunks = []
EX_CHUNK = 2000  # bigger chunks for extraction = more context per LLM call
EX_OVERLAP = 400
for i in range(0, len(document), EX_CHUNK - EX_OVERLAP):
    extract_chunks.append(document[i:i+EX_CHUNK])

print(f"Extraction: {len(extract_chunks)} chunks of {EX_CHUNK} chars (overlap {EX_OVERLAP})")
print("This is how real Graph RAG works — extract per chunk, then merge.\n")

# ── Extract from each chunk ───────────────────────────────────────
structured_llm = llm.with_structured_output(ChunkKG)

all_entities = []
all_relationships = []

EXTRACT_SYSTEM = """Extract ALL entities and relationships from this text chunk.
Entity types: PERSON, COMPANY, PRODUCT, HOSPITAL, INVESTOR, CITY, UNIVERSITY, TECHNOLOGY
Be VERY thorough — extract every person mentioned, every company, every investor,
every product, every city. Include career moves (PREVIOUSLY_AT, LEFT_FOR),
funding (INVESTED_IN, LED_ROUND), board roles (BOARD_MEMBER_OF, STEPPED_DOWN_FROM),
lawsuits (SUED, SETTLED_WITH), acquisitions (ACQUIRED), partnerships (PARTNERED_WITH),
co-authoring (CO_AUTHORED_WITH), founding (FOUNDED, CO_FOUNDED), hiring (HIRED_AT).
Do NOT skip minor details like angel investors, law firms, or one-line mentions."""

for i, chunk in enumerate(extract_chunks):
    print(f"  Chunk {i+1}/{len(extract_chunks)}...", end=" ")
    kg_chunk = structured_llm.invoke([
        SystemMessage(content=EXTRACT_SYSTEM),
        HumanMessage(content=f"Extract from this chunk:\n\n{chunk}"),
    ])
    
    for e in kg_chunk.entities:
        all_entities.append(e.model_dump())
    for r in kg_chunk.relationships:
        all_relationships.append(r.model_dump())
    
    print(f"{len(kg_chunk.entities)} entities, {len(kg_chunk.relationships)} rels")

print(f"\nTotal extracted (before merge): {len(all_entities)} entities, {len(all_relationships)} relationships")

# ── Merge duplicate entities ──────────────────────────────────────
merged = {}
for e in all_entities:
    key = e["name"].strip().lower()
    if key in merged:
        if len(e["description"]) > len(merged[key]["description"]):
            merged[key]["description"] = e["description"]
    else:
        merged[key] = e

unique_entities = list(merged.values())
print(f"After merge: {len(unique_entities)} unique entities")

# ── Deduplicate relationships ─────────────────────────────────────
seen_rels = set()
unique_rels = []
for r in all_relationships:
    key = (r["source"].strip().lower(), r["target"].strip().lower(), r["type"].strip().lower())
    if key not in seen_rels:
        seen_rels.add(key)
        unique_rels.append(r)

print(f"After dedup: {len(unique_rels)} unique relationships")

# ── Load into Neo4j with MERGE ────────────────────────────────────
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    
    for e in unique_entities:
        s.run("MERGE (n:Entity {name: $n}) SET n.type=$t, n.description=$d",
              n=e["name"], t=e["type"], d=e["description"])
    
    loaded = 0
    for r in unique_rels:
        result = s.run(
            """MATCH (a:Entity {name:$s}), (b:Entity {name:$t})
               MERGE (a)-[:RELATES_TO {type:$r, description:$d}]->(b)
               RETURN count(*) AS c""",
            s=r["source"], t=r["target"], r=r["type"], d=r["description"])
        if result.single()["c"] > 0:
            loaded += 1
    
    nodes = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    edges = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
driver.close()

print(f"\nNeo4j loaded: {nodes} nodes, {edges} relationships")
print(f"(vs Classic RAG which only sees {TOP_K} x {CHUNK_SIZE} = {TOP_K*CHUNK_SIZE} chars per question)")

---
## The Comparison Engine

Same question to both systems, answers shown side by side.

In [ ]:
COLORS = {
    "PERSON": "#FF6B6B", "COMPANY": "#4ECDC4", "PRODUCT": "#45B7D1",
    "HOSPITAL": "#98FB98", "INVESTOR": "#F39C12", "CITY": "#FFEAA7",
    "UNIVERSITY": "#96CEB4", "ROLE": "#D3D3D3", "TECHNOLOGY": "#DDA0DD",
}

all_results = []


def classic_rag(question):
    q_emb = embeddings.embed_query(question)
    results = collection.query(query_embeddings=[q_emb], n_results=TOP_K)
    chunks_used = results["documents"][0]
    context = "
---
".join(chunks_used)
    answer = ask_llm(
        f"Context:
{context}

Question: {question}",
        system="Answer ONLY from context. If info missing say so. Do NOT guess."
    )
    return answer, chunks_used


def graph_rag_query(question):
    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    # Extract entity names from question
    terms_str = ask_llm(
        f"List the key entity names (people, companies, places) from this question, one per line:
{question}",
        system="Return ONLY entity names, one per line. No explanations."
    )
    terms = [t.strip().strip("-•* ") for t in terms_str.strip().split("
") if t.strip()]
    
    # Fetch only relevant neighborhood
    triples = []
    with driver.session() as s:
        for term in terms:
            for r in s.run(
                """MATCH (a:Entity)-[r:RELATES_TO]->(b:Entity)
                   WHERE toLower(a.name) CONTAINS toLower()
                      OR toLower(b.name) CONTAINS toLower()
                   RETURN a.name AS src, r.type AS rel, b.name AS tgt""", t=term):
                triples.append(f"{r['src']} --[{r['rel']}]--> {r['tgt']}")
    driver.close()
    triples = list(set(triples))
    
    ctx = "
".join(triples)
    answer = ask_llm(
        f"Knowledge Graph:
{ctx}

Question: {question}",
        system="Answer using ONLY the graph relationships. Trace paths between entities."
    )
    return answer, triples, terms


def compare(question, q_num, difficulty):
    rag_answer, rag_chunks = classic_rag(question)
    kg_answer, kg_triples, kg_terms = graph_rag_query(question)
    
    all_results.append({
        "q_num": q_num, "difficulty": difficulty, "question": question,
        "rag_answer": rag_answer, "kg_answer": kg_answer,
        "rag_chunks": len(rag_chunks), "kg_triples": len(kg_triples),
    })
    
    print(f"
{'━'*70}")
    print(f"  Q{q_num} [{difficulty}]: {question}")
    print(f"{'━'*70}")
    
    print(f"
┌{'─'*68}┐")
    print(f"│ {'CLASSIC RAG':^66} │")
    print(f"│ {'(retrieved ' + str(len(rag_chunks)) + ' chunks, ' + str(CHUNK_SIZE) + ' chars each)':^66} │")
    print(f"├{'─'*68}┤")
    for line in rag_answer.split('. ')[:5]:
        line = line.strip()[:66]
        if line:
            print(f"│ {line:<66} │")
    print(f"└{'─'*68}┘")
    
    print(f"
┌{'─'*68}┐")
    print(f"│ {'GRAPH RAG':^66} │")
    print(f"│ {'(entities: ' + str(kg_terms) + ')':^66} │")
    print(f"│ {'(' + str(len(kg_triples)) + ' relevant triples fetched)':^66} │")
    print(f"├{'─'*68}┤")
    print(f"│ {'Subgraph:':<66} │")
    for t in kg_triples[:5]:
        t = t[:64]
        print(f"│   {t:<64} │")
    if len(kg_triples) > 5:
        print(f"│   {'... and ' + str(len(kg_triples)-5) + ' more':<64} │")
    print(f"│{'':68}│")
    print(f"│ {'Answer:':<66} │")
    for line in kg_answer.split('. ')[:5]:
        line = line.strip()[:66]
        if line:
            print(f"│ {line:<66} │")
    print(f"└{'─'*68}┘")

print("Comparison engine ready!")

## Round 1: Simple Fact

Both should handle this — answer is in a single chunk.

In [ ]:
compare("When was Meridian Health Technologies founded by Amara Osei and James Chen?", 1, "EASY")

## Round 2: Relationship (spans 3 sections)

J&J collaboration, declined co-founder offer, patent lawsuit, reconciliation.

In [ ]:
compare("Trace the complete history between Amara Osei and Robert Zhang — from Johnson & Johnson to reconciliation.", 2, "MEDIUM")

## Round 3: Career Path (spans 4 sections)

IBM Watson Health → Flatiron Health → Meridian CTO → left for Recursion.

In [ ]:
compare("Trace Lars Eriksson career from IBM Watson Health to Flatiron Health to Meridian to Recursion Pharmaceuticals.", 3, "HARD")

## Round 4: Cross-Section Aggregation

Seed (M), Series A (M), Series B (M) — each in a different chapter.

In [ ]:
compare("Who invested in Meridian — Atlas Venture, Andreessen Horowitz, SoftBank? What were the round sizes?", 4, "HARD")

## Round 5: Ironic Journey (4-hop path)

Atlas Venture → Meridian board → conflict of interest → BioNexus board.

In [ ]:
compare("Trace Priya Mehta journey from Atlas Venture board member to stepping down to joining BioNexus AI board.", 5, "VERY HARD")

## Round 6: Who Left Where? (scattered across document)

Lars, Maya, Henrik, Rachel — each departure is in a different chapter.

In [ ]:
compare("Where did Lars Eriksson, Maya Patel, Henrik Johansson, and Rachel Torres go after leaving Meridian?", 6, "VERY HARD")

## Round 7: 4-Hop Path

Thomas Lee → invested in Meridian → Meridian vs BioNexus → BioNexus acquired by Illumina.

In [ ]:
compare("What path connects angel investor Thomas Lee to the Illumina acquisition of BioNexus AI?", 7, "VERY HARD")

---
## Visualize the FULL Knowledge Graph

In [ ]:
net = Network(height="900px", width="100%", directed=True,
              bgcolor="#0d1117", font_color="white", cdn_resources="remote")
net.set_options("""
{
  "physics": {"barnesHut": {"gravitationalConstant": -6000, "springLength": 200, "springConstant": 0.03},
              "stabilization": {"iterations": 150}},
  "edges": {"color": {"color": "#636e72", "highlight": "#e17055"},
            "arrows": {"to": {"scaleFactor": 0.7}},
            "font": {"size": 9, "color": "#b2bec3", "strokeWidth": 0},
            "smooth": {"type": "curvedCW", "roundness": 0.15}},
  "nodes": {"font": {"size": 13, "color": "white", "face": "arial"},
            "borderWidth": 2, "borderWidthSelected": 4},
  "interaction": {"hover": true, "tooltipDelay": 100, "zoomView": true}
}
""")

SHAPES = {"PERSON": "dot", "COMPANY": "diamond", "PRODUCT": "star",
          "HOSPITAL": "triangle", "INVESTOR": "square", "CITY": "triangleDown"}

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    degrees = {}
    for r in s.run("MATCH (n) OPTIONAL MATCH (n)-[r]-() RETURN n.name AS name, count(r) AS deg"):
        degrees[r["name"]] = r["deg"]
    
    for r in s.run("MATCH (n:Entity) RETURN n.name AS name, n.type AS type, n.description AS desc"):
        color = COLORS.get(r["type"], "#DFE6E9")
        shape = SHAPES.get(r["type"], "dot")
        size = 12 + degrees.get(r["name"], 0) * 4
        tooltip = f"<b>{r['name']}</b><br>Type: {r['type']}<br>{r['desc']}"
        net.add_node(r["name"], label=r["name"], color=color, shape=shape,
                     size=size, title=tooltip)
    
    for r in s.run("MATCH (a)-[r]->(b) RETURN a.name AS s, b.name AS t, r.type AS type, r.description AS d"):
        net.add_edge(r["s"], r["t"], label=r["type"], color="#636e72",
                     title=r["d"], font={"size": 9, "color": "#ffeaa7"})
driver.close()

html_path = os.path.abspath("full_kg_showdown.html")
net.save_graph(html_path)
print(f"Open in browser: {html_path}")

## Final Scorecard

Run this after all 7 rounds to see the summary.

In [ ]:
print(f"\n{'━'*70}")
print(f"  FINAL SCORECARD — Classic RAG vs Graph RAG")
print(f"{'━'*70}\n")

print(f"{'Q#':<4} {'Difficulty':<12} {'RAG Chunks':<12} {'KG Triples':<12} Question")
print(f"{'─'*4} {'─'*12} {'─'*12} {'─'*12} {'─'*30}")
for r in all_results:
    print(f"Q{r['q_num']:<3} {r['difficulty']:<12} {r['rag_chunks']:<12} {r['kg_triples']:<12} {r['question'][:40]}")

print(f"\n{'━'*70}")
print(f"  WHEN TO USE WHAT")
print(f"{'━'*70}")
print(f"""
  Classic RAG wins when:
    - Answer is in a single chunk (simple facts)
    - No cross-references needed
    - Speed matters more than completeness

  Graph RAG wins when:
    - Answer spans multiple sections (career paths, funding history)
    - Relationships between entities matter (who connected to whom)
    - Indirect/multi-hop connections (A → B → C → D)
    - Aggregation across the document (list ALL of something)
    - Temporal tracking (what changed over time)

  Best practice: Use BOTH together (Hybrid RAG)
    - Vector search for quick fact lookup
    - Graph traversal for relationship questions
""")